# Feature selection as a validated decision
Selection can reduce noise, cost and instability, but selecting on the full dataset leaks information. The selector must live inside CV.

In [ ]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, mutual_info_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

X,y=make_classification(n_samples=1800,n_features=80,n_informative=10,n_redundant=10,random_state=42)
cv=StratifiedKFold(5,shuffle=True,random_state=42)
models={
 'all_features':make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000)),
 'kbest_20':make_pipeline(SelectKBest(mutual_info_classif,k=20),StandardScaler(),LogisticRegression(max_iter=3000)),
 'model_select':make_pipeline(SelectFromModel(RandomForestClassifier(n_estimators=120,random_state=42),threshold='median'),StandardScaler(),LogisticRegression(max_iter=3000)),
}
pd.Series({n:cross_val_score(m,X,y,cv=cv,scoring='roc_auc').mean() for n,m in models.items()},name='cv_auc').round(4)


## Interpretation
A smaller feature set is valuable only if it improves or preserves out-of-sample utility while reducing complexity/cost. Feature selection is not an aesthetic cleanup step.